In [ ]:
import subprocess, sys

# Uninstall everything ORT-related
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "onnxruntime-gpu", "onnxruntime"], capture_output=True)

# Install ORT built for CUDA 12.x
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "onnxruntime-gpu==1.20.1",
                "--extra-index-url",
                "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/"],
               check=True)

print("ORT installed. NOW: Runtime → Restart session")
print("After restart skip this cell and run from Cell 1.")

ORT installed. NOW: Runtime → Restart session
After restart skip this cell and run from Cell 1.


In [ ]:
import subprocess, sys, os, re
from pathlib import Path

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("ultralytics", "scipy", "matplotlib", "tqdm", "kaggle", "opencv-python-headless")
pip("onnx>=1.14.0", "onnxmltools>=1.11.0", "onnxconverter-common")
# Do NOT install onnxruntime-gpu here — already installed in Cell 0

# Verify ORT loads with CUDA
import onnxruntime as ort
print(f"ORT version:  {ort.__version__}")
print(f"Providers:    {ort.get_available_providers()}")
assert "CUDAExecutionProvider" in ort.get_available_providers(), \
    "CUDA provider missing — re-run Cell 0 and restart"

# Clone and patch TrackEval
if not os.path.exists("/content/TrackEval"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/JonathonLuiten/TrackEval.git",
                    "/content/TrackEval"], check=True)

fixes = {
    "np.float)": "np.float64)", "np.float,": "np.float64,",
    "np.int)":   "np.int64)",   "np.int,":   "np.int64,",
    "np.bool)":  "np.bool_)",   "np.bool,":  "np.bool_,",
}
for f in Path("/content/TrackEval").rglob("*.py"):
    txt = f.read_text(); new = txt
    for old, repl in fixes.items():
        new = new.replace(old, repl)
    new = re.sub(r"np\.bool_\(\)", "np.bool_", new)
    if new != txt: f.write_text(new)

import torch
print(f"torch: {torch.__version__}  CUDA: {torch.cuda.is_available()}")
print("Dependencies Ready")

ORT version:  1.20.1
Providers:    ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
torch: 2.11.0+cu128  CUDA: True
Dependencies Ready


In [ ]:
import json, os, subprocess
from google.colab import files

print("Upload: config.py, export.py, benchmark.py, accuracy.py, profile.py")
uploaded = files.upload()
print(f"Uploaded: {list(uploaded.keys())}")

# Download MOT17
KAGGLE_USERNAME = "jesonr"   # ← fill in
KAGGLE_KEY      = "KGAT_d385003cc32ed4431502e2a6c631e34a"        # ← fill in

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

from pathlib import Path
if not Path("/content/MOT17/train").exists():
    subprocess.run(["kaggle", "datasets", "download",
                    "-d", "wenhoujinjust/mot-17",
                    "-p", "/content/", "--unzip"], check=True)
print("MOT17 ready")

Upload: config.py, export.py, benchmark.py, accuracy.py, profile.py


Saving benchmark.py to benchmark.py
Saving profile.py to profile.py
Saving accuracy.py to accuracy.py
Saving export.py to export.py
Saving config.py to config.py
Uploaded: ['benchmark.py', 'profile.py', 'accuracy.py', 'export.py', 'config.py']
MOT17 ready


In [ ]:
# Fix onnxmltools FP16 converter import path
import onnxmltools
print(f"onnxmltools version: {onnxmltools.__version__}")

# Find where the converter actually lives in this version
import subprocess
result = subprocess.run(
    ["python3", "-c", "import onnxmltools; import os; print(os.path.dirname(onnxmltools.__file__))"],
    capture_output=True, text=True
)
onnxmltools_path = result.stdout.strip()
print(f"onnxmltools path: {onnxmltools_path}")

import os
for root, dirs, files in os.walk(onnxmltools_path):
    for f in files:
        if "float16" in f.lower() or "converter" in f.lower():
            rel = os.path.relpath(os.path.join(root, f), onnxmltools_path)
            print(f"  Found: {rel}")

onnxmltools version: 1.16.0
onnxmltools path: /usr/local/lib/python3.12/dist-packages/onnxmltools
  Found: convert/libsvm/operator_converters/SVMConverter.py
  Found: convert/libsvm/operator_converters/__pycache__/SVMConverter.cpython-312.pyc


In [ ]:
%run export.py

Export dependencies installed ✓
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Exporting YOLOv8x to ONNX FP32...
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs

PyTorch: starting from 'yolov8x.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (130.5 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 283ms
Prepared 2 packages i

/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 7.951847180720506e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -9.991038041334832e-08 will be truncated to -1e-07
  warnings.warn(


  FP32: 273.1 MB  →  FP16: 136.6 MB (50% smaller) ✓

Loading calibration data for INT8...
Loaded 100 calibration frames from MOT17-09-SDP ✓
Applying INT8 PTQ with 100 calibration images...


  FP32: 273.1 MB  →  INT8: 69.2 MB (75% smaller) ✓

All exports complete:
  yolov8x.onnx: 273.1 MB
  yolov8x_fp16.onnx: 136.6 MB
  yolov8x_int8.onnx: 69.2 MB


In [ ]:
import onnxruntime as ort
import onnx
import config

print(f"ORT providers: {ort.get_available_providers()}")

# Quick GPU inference test on FP32 model
session = ort.InferenceSession(
    str(config.ONNX_MODEL),
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
print(f"Active providers: {session.get_providers()}")
assert session.get_providers()[0] == "CUDAExecutionProvider", \
    "ORT not using GPU — Cell 0 fix did not take effect"

import numpy as np
inp  = np.random.rand(1, 3, 640, 640).astype(np.float32)
name = session.get_inputs()[0].name
out  = session.run(None, {name: inp})
scores = out[0][0].transpose(1, 0)[:, 4]
print(f"Test inference scores max: {scores.max():.4f}")
print("ORT GPU confirmed ✓  — proceed to Cell 5")
del session

ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Test inference scores max: 0.0001
ORT GPU confirmed ✓  — proceed to Cell 5


In [ ]:
%run benchmark.py

Loading test frame...
  Input shape: (1, 3, 640, 640)  dtype: float32

Benchmarking PyTorch FP32...
  Warmup (20 runs)...
  Timing (100 runs)...
  PyTorch FP32:
    Mean ± Std: 58.28 ± 0.47 ms
    P50 / P95 / P99: 58.32 / 59.13 / 59.58 ms
    FPS: 17.2

Benchmarking PyTorch FP16...
  Warmup (20 runs)...
  Timing (100 runs)...
  PyTorch FP16:
    Mean ± Std: 22.82 ± 0.87 ms
    P50 / P95 / P99: 22.74 / 23.19 / 28.55 ms
    FPS: 43.8

Benchmarking ONNX FP32...
  Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Warmup (20 runs)...
  Timing (100 runs)...
  ONNX FP32:
    Mean ± Std: 63.53 ± 0.73 ms
    P50 / P95 / P99: 63.49 / 64.48 / 65.21 ms
    FPS: 15.7

Benchmarking ONNX FP16...
  Skipping ONNX FP16 — failed to load model: [ONNXRuntimeError] : 1 : FAIL : Load model from /content/models/yolov8x_fp16.onnx failed:Type Error: Type (tensor(float)) of output arg (/model.10/Resize_output_0) of node (/model.10/Resize_output_cast0) does not match expected type (tensor(floa

In [ ]:
%run accuracy.py


Running pipeline: YOLOv8x-FP32


  YOLOv8x-FP32: 100%|██████████| 525/525 [00:43<00:00, 12.10it/s]


  YOLOv8x-FP32: 3338 track rows | 12.1 FPS (end-to-end)
  Evaluating HOTA...

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : False                         
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : False                         
DISPLAY_LESS_PROGRESS : True                          
OUTPUT_SUMMARY       : False                         
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : False                         
PLOT_CURVES          : False                         

MotChallenge2DBox Config:
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mot

  YOLOv8x-INT8: 100%|██████████| 525/525 [00:59<00:00,  8.86it/s]


  YOLOv8x-INT8: 0 track rows | 8.9 FPS (end-to-end)
  Evaluating HOTA...

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : False                         
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : False                         
DISPLAY_LESS_PROGRESS : True                          
OUTPUT_SUMMARY       : False                         
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : False                         
PLOT_CURVES          : False                         

MotChallenge2DBox Config:
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mot_cha

In [ ]:
%run profile.py

Mode A: Manual stage timing
Profiling 50 frames (manual timing)...

PIPELINE STAGE BREAKDOWN (mean ms per frame, FP32)
  Preprocessing              15.32 ms   17.7%  ████████
  Backbone (approx)          39.58 ms   45.8%  ██████████████████████
  Neck/PAN (approx)          13.19 ms   15.3%  ███████
  Head (approx)              13.19 ms   15.3%  ███████
  Postprocess (NMS)           5.12 ms    5.9%  ██
  ByteTrack (assoc)           0.06 ms    0.1%  
───────────────────────────────────────────────────────
  Total pipeline             86.46 ms  → 11.6 FPS

  Note: backbone/neck/head split is approximate (60/20/20%)
  Chrome trace gives exact kernel-level breakdown.

Mode B: PyTorch Profiler (Chrome trace)
Running PyTorch Profiler on 20 frames...
  (CUDA activities, CPU ops, memory)


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


  PyTorch Profiler failed: Trace is already saved.
  Manual timings above are still valid.

Profile summary saved to /content/benchmark_results/profile_summary.json


In [ ]:
import json
from pathlib import Path

results_dir = Path("/content/benchmark_results")

print("── Latency ──────────────────────────────────────────")
with open(results_dir / "latency_benchmark.json") as f:
    lat = json.load(f)
for key, r in lat.items():
    print(f"  {r['label']:<22} {r['mean_ms']:>8.2f} ms  {r['fps']:>6.1f} FPS")

print("\n── Accuracy (MOT17-09-SDP) ──────────────────────────")
with open(results_dir / "accuracy_results.json") as f:
    acc = json.load(f)
for key, r in acc.items():
    print(f"  {r['tracker_name']:<22} "
          f"HOTA={r['HOTA']}  MOTA={r['MOTA']}  IDF1={r['IDF1']}")

print("\n── Pipeline stages ──────────────────────────────────")
with open(results_dir / "profile_summary.json") as f:
    prof = json.load(f)
for stage, ms in prof["manual_timings_ms"].items():
    if stage in ["preprocess", "model_forward", "postprocess",
                 "bytetrack_hungarian", "total_pipeline"]:
        print(f"  {stage:<30} {ms:>7.2f} ms")

── Latency ──────────────────────────────────────────
  PyTorch FP32              58.28 ms    17.2 FPS
  PyTorch FP16              22.82 ms    43.8 FPS
  ONNX FP32                 63.53 ms    15.7 FPS
  ONNX INT8                 88.77 ms    11.3 FPS

── Accuracy (MOT17-09-SDP) ──────────────────────────
  YOLOv8x-FP32           HOTA=56.07  MOTA=55.55  IDF1=56.1
  YOLOv8x-INT8           HOTA=0.0  MOTA=0.0  IDF1=0.0

── Pipeline stages ──────────────────────────────────
  preprocess                       15.32 ms
  model_forward                    65.96 ms
  postprocess                       5.12 ms
  bytetrack_hungarian               0.06 ms
  total_pipeline                   86.46 ms
